In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from src.logger import logging
import os
import time
import mlflow
import scipy.sparse
import dagshub



In [2]:
df = pd.read_csv(r'D:\Resume-Screening-Matching-System\data_upload_S3\data\cleaned_dataset.csv')
df.head()

,Role,Resume
0,E-commerce Specialist,Here's a professional resume for Jason Jones:\...
1,Game Developer,Here's a professional resume for Ann Marshall:...
2,Human Resources Specialist,Here's a professional resume for Patrick Mccla...
3,E-commerce Specialist,Here's a professional resume for Patricia Gray...
4,E-commerce Specialist,Here's a professional resume for Amanda Gross:...


Preprocessing Data:

In [3]:
def clean_text(text):
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove URLs (linkedin, github, websites)
    text = re.sub(r'http\S+|www\.\S+|linkedin\.com/\S+|github\.com/\S+', ' ', text)

    # Remove phone numbers
    text = re.sub(
        r'(\+\d{1,3}[-.\s]?)?(\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4})',
        ' ',
        text
    )

    # Remove markdown links
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)

    # Remove bullet symbols
    text = re.sub(r'[•▪◦●■♦★]', ' ', text)

    # Remove asterisks used as bullets
    text = re.sub(r'\*', ' ', text)

    # Keep only letters, numbers, +, #, /, . and spaces
    text = re.sub(r'[^a-zA-Z0-9+#/. ]', ' ', text)

    # Convert to lowercase
    text = text.lower()

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    

    return text

def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def normalize_text(text):
    """Normalize the text by applying all preprocessing steps."""
    text = clean_text(text)
    text = lemmatization(text)
    text = remove_stop_words(text)
    return text

In [4]:
df['Resume'] = df['Resume'].apply(lambda x: normalize_text(x))

In [5]:
df['Resume'][0]

'professional resume jason jones jason jones e commerce specialist contact information email phone linkedin summary result driven e commerce specialist 5+ year experience inventory management seo online advertising analytics. proven track record increasing online sale improving website traffic optimizing inventory levels. skilled analyzing complex data set identifying trend making data driven decisions. passionate staying date latest e commerce trend technologies. professional experience e commerce specialist xyz corporation 2018 present managed inventory level across multiple channel resulting 25 reduction stockouts 15 reduction overstocking developed implemented seo strategy increased website traffic 30 improved search engine ranking 20 created executed online advertising campaign generated 50 increase sale 20 increase conversion rate analyzed website analytics identify trend optimize user experience improve customer engagement collaborated cross functional team launch new product li

Target Column into Categorical values

In [6]:
label = LabelEncoder()
df['Role'] = label.fit_transform(df['Role'])
df.head()

,Role,Resume
0,15,professional resume jason jones jason jones e ...
1,17,professional resume ann marshall ann marshall ...
2,18,professional resume patrick mcclain patrick mc...
3,15,professional resume patricia gray patricia gra...
4,15,professional resume amanda gross amanda gross ...


In [7]:
VECTORIZERS = {
    'BoW': CountVectorizer(max_features=200, ngram_range=(1,2)),
    'TF-IDF': TfidfVectorizer(max_features=200, ngram_range=(1,2))
}

ALGORITHMS = {
    'LogisticRegression': LogisticRegression(),
    'MultinomialNB': MultinomialNB(),
    'RandomForest': RandomForestClassifier(),
    'LinearSVC': LinearSVC()
}


In [8]:
def log_model_params(algo_name, model):
    """Logs hyperparameters of the trained model to MLflow."""
    params_to_log = {}
    if algo_name == 'LogisticRegression':
        params_to_log["C"] = model.C
    elif algo_name == 'MultinomialNB':
        params_to_log["alpha"] = model.alpha
    elif algo_name == 'RandomForest':
        params_to_log["n_estimators"] = model.n_estimators
        params_to_log["max_depth"] = model.max_depth
    elif algo_name == 'LinearSVC':
        params_to_log["C"] = model.C
    

    mlflow.log_params(params_to_log)

In [13]:
CONFIG = {
    "data_path": "notebooks/data.csv",
    "test_size": 0.2,
    "mlflow_tracking_uri": "https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow",
    "dagshub_repo_owner": "mdzaid382",
    "dagshub_repo_name": "Resume-Screening-Matching-System",
    "experiment_name": "Bow vs Tf-Idf"
}

mlflow.set_tracking_uri(CONFIG["mlflow_tracking_uri"])
dagshub.init(repo_owner=CONFIG["dagshub_repo_owner"], repo_name=CONFIG["dagshub_repo_name"], mlflow=True)
mlflow.set_experiment(CONFIG["experiment_name"])

with mlflow.start_run(run_name="All Experiments") as parent_run:
    for algo_name, algorithm in ALGORITHMS.items():
        for vec_name, vectorizer in VECTORIZERS.items():
            with mlflow.start_run(run_name=f"{algo_name} with {vec_name}", nested=True) as child_run:
                try:
                    
                    logging.info("Initializing train test split...")
                    X_train, X_test, y_train, y_test = train_test_split(df['Resume'], df['Role'], test_size=CONFIG["test_size"], random_state=42)

                    logging.info("Vectorizing text data...")
                    X_train = vectorizer.fit_transform(X_train)
                    X_test = vectorizer.transform(X_test)
                    
                    logging.info("Logging preprocessing parameters...")
                    mlflow.log_params({
                        "vectorizer": vec_name,
                        "algorithm": algo_name,
                        "test_size": CONFIG["test_size"]
                    })

                    logging.info("Training model...")
                    model = algorithm
                    model.fit(X_train, y_train)

                    logging.info("Logging model parameters...")
                    log_model_params(algo_name, model)
                   
                    logging.info("Evaluating model...")
                    y_pred = model.predict(X_test)

                    metrics = {
                        "accuracy": accuracy_score(y_test, y_pred,),
                        "precision": precision_score(y_test, y_pred, average='weighted'),
                        "recall": recall_score(y_test, y_pred, average='weighted'),
                        "f1_score": f1_score(y_test, y_pred, average='weighted')
                    }

                    logging.info("Logging evaluation metrics...")
                    mlflow.log_metrics(metrics)

                    logging.info("Logging model artifact...")
                    input_example = X_test[:5] if not scipy.sparse.issparse(X_test) else X_test[:5].toarray()
                    mlflow.sklearn.log_model(model, "model", input_example=input_example)
                    # Print results for verification
                    print(f"\nAlgorithm: {algo_name}, Vectorizer: {vec_name}")
                    print(f"Metrics: {metrics}")

                except Exception as e:
                    print(f"Error in training {algo_name} with {vec_name}: {e}")
                    mlflow.log_param("error", str(e))




[ 2026-06-03 21:48:20,279 ] httpx - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/mdzaid382/Resume-Screening-Matching-System "HTTP/1.1 200 OK"


Initialized MLflow to track repo "mdzaid382/Resume-Screening-Matching-System"

[ 2026-06-03 21:48:20,283 ] dagshub - INFO - Initialized MLflow to track repo "mdzaid382/Resume-Screening-Matching-System"


Repository mdzaid382/Resume-Screening-Matching-System initialized!

[ 2026-06-03 21:48:20,286 ] dagshub - INFO - Repository mdzaid382/Resume-Screening-Matching-System initialized!


...


Algorithm: MultinomialNB, Vectorizer: TF-IDF
Metrics: {'accuracy': 0.9533169533169533, 'precision': 0.9578191814199982, 'recall': 0.9533169533169533, 'f1_score': 0.9464725541088086}
🏃 View run MultinomialNB with TF-IDF at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/7/runs/36e2353178e04d1e964c22d36f351cea
🧪 View experiment at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/7
[ 2026-06-03 21:55:52,691 ] root - INFO - Initializing train test split...
[ 2026-06-03 21:55:52,697 ] root - INFO - Vectorizing text data...
[ 2026-06-03 21:55:57,054 ] root - INFO - Logging preprocessing parameters...
[ 2026-06-03 21:55:57,914 ] root - INFO - Training model...
[ 2026-06-03 21:56:06,387 ] root - INFO - Logging model parameters...
[ 2026-06-03 21:56:06,925 ] root - INFO - Evaluating model...
[ 2026-06-03 21:56:07,034 ] root - INFO - Logging evaluation metrics...
[ 2026-06-03 21:56:09,282 ] root - INFO - Logging model arti


Algorithm: RandomForest, Vectorizer: BoW
Metrics: {'accuracy': 0.995085995085995, 'precision': 0.9951231621231621, 'recall': 0.995085995085995, 'f1_score': 0.9950859765803594}
🏃 View run RandomForest with BoW at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/7/runs/449196ef5cb2443588ffa86b6f44122e
🧪 View experiment at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/7
[ 2026-06-03 21:57:47,693 ] root - INFO - Initializing train test split...
[ 2026-06-03 21:57:47,702 ] root - INFO - Vectorizing text data...
[ 2026-06-03 21:57:52,281 ] root - INFO - Logging preprocessing parameters...
[ 2026-06-03 21:58:02,950 ] urllib3.connectionpool - WARNING - Retrying (Retry(total=6, connect=7, read=6, redirect=7, status=7)) after connection broken by 'RemoteDisconnected('Remote end closed connection without response')': /mdzaid382/Resume-Screening-Matching-System.mlflow/api/2.0/mlflow/runs/log-batch
[ 2026-06-03 21:58:05,200


Algorithm: RandomForest, Vectorizer: TF-IDF
Metrics: {'accuracy': 0.9921375921375921, 'precision': 0.9922328530757321, 'recall': 0.9921375921375921, 'f1_score': 0.9921260317339621}
🏃 View run RandomForest with TF-IDF at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/7/runs/269b437296304b7991d27ddb3903ab8f
🧪 View experiment at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/7
[ 2026-06-03 22:04:12,114 ] urllib3.connectionpool - WARNING - Retrying (Retry(total=6, connect=7, read=6, redirect=7, status=7)) after connection broken by 'ReadTimeoutError("HTTPSConnectionPool(host='dagshub.com', port=443): Read timed out. (read timeout=120)")': /mdzaid382/Resume-Screening-Matching-System.mlflow/api/2.0/mlflow/runs/update
[ 2026-06-03 22:04:19,434 ] root - INFO - Initializing train test split...
[ 2026-06-03 22:04:19,443 ] root - INFO - Vectorizing text data...
[ 2026-06-03 22:04:23,771 ] root - INFO - Logging preprocess


Algorithm: LinearSVC, Vectorizer: BoW
Metrics: {'accuracy': 0.9921375921375921, 'precision': 0.9922931850158989, 'recall': 0.9921375921375921, 'f1_score': 0.9921321772520943}
🏃 View run LinearSVC with BoW at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/7/runs/b9a5859bf44a4c29814b44c1f310285a
🧪 View experiment at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/7
[ 2026-06-03 22:05:18,262 ] root - INFO - Initializing train test split...
[ 2026-06-03 22:05:18,267 ] root - INFO - Vectorizing text data...
[ 2026-06-03 22:05:22,729 ] root - INFO - Logging preprocessing parameters...
[ 2026-06-03 22:05:23,178 ] root - INFO - Training model...
[ 2026-06-03 22:05:24,415 ] root - INFO - Logging model parameters...
[ 2026-06-03 22:05:25,634 ] root - INFO - Evaluating model...
[ 2026-06-03 22:05:25,652 ] root - INFO - Logging evaluation metrics...
[ 2026-06-03 22:05:26,147 ] root - INFO - Logging model artifact...



Algorithm: LinearSVC, Vectorizer: TF-IDF
Metrics: {'accuracy': 0.9945945945945946, 'precision': 0.994780600776113, 'recall': 0.9945945945945946, 'f1_score': 0.9945885413146585}
🏃 View run LinearSVC with TF-IDF at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/7/runs/36072ec7c2bd46059fba9fa509ecfcb2
🧪 View experiment at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/7
🏃 View run All Experiments at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/7/runs/4b538e600f244282b7f969fcf3fd219e
🧪 View experiment at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/7
